# Bench 实验
从上到下运行；标「可重跑」的格可以单独再跑。ESP 每次 reset 时保持线圈电源关闭。
结束后：`uv run python controller/control/safe_off.py`，关电源。步骤说明见 `README.md`。

## 导入

In [5]:
from pathlib import Path
import sys
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'LLM/bench.py').exists())
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
from IPython.display import display, Image
from LLM.bench import Bench

## 设置
改任何一项都从「建立 bench」重跑。`CALIBRATION = None` 表示重新标定；`PLUMB = None` 表示不做装置定位（sync check）。

In [6]:
CAMERAS = (1, 0)                                   # (A, B) 相机序号
PORT = '/dev/cu.usbserial-0001'                    # ESP 串口
CALIBRATION = PROJECT/'LLM/runs/setup_002/calibration/stereo_rig.json'  # None = 重新标定
PLUMB = None   # 装置定位用的垂线像素 {'bottom': {'a': [x, y], 'b': [x, y]}, 'top': {...}}；None = sync check
RAMP = 'alignment_50Hz'                            # LLM/ramps/ 中的文件名
NOTE = ''                                          # robot / bearing / 线圈、实验目的
print('可选 ramp:', sorted(p.stem for p in (PROJECT/'LLM/ramps').glob('*.json')))

可选 ramp: ['alignment_50Hz', 'linear_to_190Hz_hold', 'linear_to_250Hz', 'two_step_linear_to_190Hz_hold']


## 0 · 建立 bench
新 session；之前的验证全部作废。

In [7]:
bench = Bench(cameras=CAMERAS, serial_port=PORT)

Session: /Users/meli/Desktop/Kevin/UCB/FlyingRobotsLinLiweiLab/ESP32_PMW/LLM/runs/sessions/20261104_112316_18a9a8


## 1 · 相机预览：对准、对焦、补光（可重跑）
画面直接显示在本格输出里（A | B），不录像。参数都在本格开头，改了重跑本格即可；中断 kernel 结束预览，相机自动恢复自动曝光。

- 遮挡一个镜头确认 A/B。
- 调相机朝向、对焦和灯光，直到两机 mean ≈ 59/82、max ≈ 235；mean 低于 ~20 时检测不到板子。
- `ROTATE_180`：每台相机单独转 180°，只能 0° 或 180°；其他角度靠转动相机本身。改变会**新建 session 并清空标定**：之后要设 `CALIBRATION = None` 重新标定，或载入在同一朝向下拍的标定。
- `EXPOSURE_MS`：手动曝光，第 2、3 步拍板子用同一个值，所以这里看到的就是标定会保存的画面。`None` = 相机自动。
- 标定之后再动相机或对焦，都要重新标定。

In [15]:
# ---- 本格参数 ----
ROTATE_180 = (True, True)   # (A, B) 是否旋转 180°；改变会新建 session、清空标定
EXPOSURE_MS = 50.0           # 手动曝光（ms），标定拍板子用同一个值；None = 相机自动
SCALE = 0.5                 # 预览缩放，只影响显示大小

bench.preview_cameras(rotate180=ROTATE_180, exposure_ms=EXPOSURE_MS, scale=SCALE)

Image(value=b'', format='jpeg')

cam0: mean  47.9 max 255  cam1: mean  37.6 max 255
cam0: mean  48.1 max 255  cam1: mean  37.7 max 255
cam0: mean  48.2 max 255  cam1: mean  37.6 max 255
cam0: mean  48.0 max 255  cam1: mean  37.7 max 255
cam0: mean  48.1 max 255  cam1: mean  37.7 max 255
cam0: mean  48.2 max 255  cam1: mean  37.6 max 255
cam0: mean  48.0 max 255  cam1: mean  37.7 max 255
cam0: mean  48.2 max 255  cam1: mean  37.7 max 255
cam0: mean  48.2 max 255  cam1: mean  37.6 max 255
cam0: mean  48.0 max 255  cam1: mean  37.7 max 255
cam0: mean  48.2 max 255  cam1: mean  37.7 max 255
cam0: mean  48.1 max 255  cam1: mean  37.6 max 255
cam0: mean  48.0 max 255  cam1: mean  37.7 max 255
cam0: mean  48.2 max 255  cam1: mean  37.7 max 255
cam0: mean  48.1 max 255  cam1: mean  37.7 max 255
cam0: mean  48.0 max 255  cam1: mean  37.7 max 255
cam0: mean  48.2 max 255  cam1: mean  37.6 max 255
cam0: mean  48.1 max 255  cam1: mean  37.7 max 255
cam0: mean  48.1 max 255  cam1: mean  37.7 max 255
cam0: mean  48.2 max 255  cam1:

## 2 · 标定（可重跑）
首次使用、重新对焦、单台移动、换模式时设 `CALIBRATION = None`。板子静止摆多个姿态，按 q 结束。

In [ ]:
if CALIBRATION is None:
    bench.collect_charuco_frames()
    bench.calibrate_intrinsics()
    bench.calibrate_extrinsics()
else:
    bench.use_calibration(CALIBRATION)

## 3 · 在最终机位验证（可重跑）
至少 5 个新的静止板子姿态，按 q 结束；须 PASS。

In [ ]:
display(bench.verify_calibration())

## 4 · 参考照片（可重跑）
移走板子；LED 与垂线两机都要看得到。点 Save，再 Close。

In [ ]:
reference = bench.photograph_reference()
for tag in 'AB':
    display(Image(filename=str(reference/f'{tag}.png'), width=800))

## 5 · 可选：装置定位（可重跑）
步骤见 `LLM/analyse/HOLDER_MARKERS.md`。`PLUMB = None` 时跳过，`run()` 做 sync check。

In [ ]:
if PLUMB is not None:
    display(bench.find_holder_marks())
    for tag in 'AB':
        display(Image(filename=str(reference/f'{tag}_holder_landmarks.png'), width=800))
    display(bench.locate_fixture(PLUMB))

## 6 · 10 秒试录（可重跑）
纯相机，自动审核。

In [ ]:
display(bench.record())

## 7 · 查看 ramp（可重跑）

In [ ]:
display(bench.show_ramp(RAMP))

## 8 · 正式实验：Record → START → 自动 stop
**会驱动线圈。** 看到 READY 后再打开线圈电源。

In [ ]:
display(bench.run(RAMP, note=NOTE))

## 9 · 分析（可重跑）
LED 对时 → frequency + 四路 CS 曲线。

In [ ]:
display(bench.analyse())
for name in ('input_curves.png', 'overview.png'):
    if (bench.last_run/'export'/name).exists():
        display(Image(filename=str(bench.last_run/'export'/name)))